In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from nltk.tokenize import WordPunctTokenizer

In [2]:
df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

In [3]:
df_train.head(3)

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1


In [4]:
tokenizer = WordPunctTokenizer()

df_train["prompt"] = df_train["prompt"].apply(lambda x: tokenizer.tokenize(str(x).lower()))
df_train["response_a"] = df_train["response_a"].apply(lambda x: tokenizer.tokenize(str(x).lower()))
df_train["response_b"] = df_train["response_b"].apply(lambda x: tokenizer.tokenize(str(x).lower()))

df_test["prompt"] = df_test["prompt"].apply(lambda x: tokenizer.tokenize(str(x).lower()))
df_test["response_a"] = df_test["response_a"].apply(lambda x: tokenizer.tokenize(str(x).lower()))
df_test["response_b"] = df_test["response_b"].apply(lambda x: tokenizer.tokenize(str(x).lower()))

In [5]:
from collections import Counter

counter = Counter()

for col in ["prompt", "response_a", "response_b"]:
    for tokens in df_train[col]:
        counter.update(tokens)

vocab = {"<PAD>": 0, "<UNK>": 1}
for token, _ in counter.items():
    vocab[token] = len(vocab)

In [6]:
def tokens_to_id(tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]


df_train["prompt_ids"] = df_train["prompt"].apply(lambda x: tokens_to_id(x, vocab))
df_train["response_a_ids"] = df_train["response_a"].apply(lambda x: tokens_to_id(x, vocab))
df_train["response_b_ids"] = df_train["response_b"].apply(lambda x: tokens_to_id(x, vocab))

df_test["prompt_ids"] = df_test["prompt"].apply(lambda x: tokens_to_id(x, vocab))
df_test["response_a_ids"] = df_test["response_a"].apply(lambda x: tokens_to_id(x, vocab))
df_test["response_b_ids"] = df_test["response_b"].apply(lambda x: tokens_to_id(x, vocab))

In [7]:
def make_label(row):
    if row["winner_model_a"] == 1:
        return 0
    elif row["winner_model_b"] == 1:
        return 1
    else:
        return 2

df_train["label"] = df_train.apply(make_label, axis=1)

In [8]:
df_train["a_len"] = df_train["response_a"].apply(lambda x: len(x))
df_train["b_len"] = df_train["response_b"].apply(lambda x: len(x))
df_train["len_diff"] = df_train["a_len"] - df_train["b_len"]
df_train["prompt_len"] = df_train["prompt"].apply(lambda x: len(x))

df_test["a_len"] = df_test["response_a"].apply(lambda x: len(x))
df_test["b_len"] = df_test["response_b"].apply(lambda x: len(x))
df_test["len_diff"] = df_test["a_len"] - df_train["b_len"]
df_test["prompt_len"] = df_test["prompt"].apply(lambda x: len(x))

In [9]:
from torch.utils.data import Dataset, DataLoader

class PreferenceDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]

        num_features = torch.tensor([
            row["prompt_len"],
            row["a_len"],
            row["b_len"],
            row["len_diff"]
        ], dtype=torch.float)

        item = {
            "prompt_ids": row["prompt_ids"],
            "a_ids": row["response_a_ids"],
            "b_ids": row["response_b_ids"],
            "numeric_features": num_features,
        }
        if not self.is_test:
            item["label"] = int(row["label"])
            
        if "id" in row.index:
            item["id"] = row["id"]

        return item

In [10]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    prompt_tensors = [torch.tensor(x["prompt_ids"], dtype=torch.long) for x in batch]
    a_tensors = [torch.tensor(x["a_ids"], dtype=torch.long) for x in batch]
    b_tensors = [torch.tensor(x["b_ids"], dtype=torch.long) for x in batch]

    prompt_ids = pad_sequence(prompt_tensors, batch_first=True, padding_value=0)
    a_ids = pad_sequence(a_tensors, batch_first=True, padding_value=0)
    b_ids = pad_sequence(b_tensors, batch_first=True, padding_value=0)

    numeric_features = torch.stack([x["numeric_features"] for x in batch])

    result = {
        "prompt_ids": prompt_ids,
        "a_ids": a_ids,
        "b_ids": b_ids,
        "numeric_features": numeric_features,
    }

    if "label" in batch[0]:
        result["label"] = torch.tensor([x["label"] for x in batch], dtype=torch.long)

    if "id" in batch[0]:
        result["id"] = [x["id"] for x in batch]

    return result

In [11]:
train_dataset = PreferenceDataset(df_train, is_test=False)
test_dataset = PreferenceDataset(df_test, is_test=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

In [12]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNNEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim=256, kernel_sizes=(3, 4, 5), dropout=0.3):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList(nn.Conv1d(emb_dim, hidden_dim, k) for k in kernel_sizes)
        self.dropout = nn.Dropout(dropout)

        self.out_dim = hidden_dim * len(kernel_sizes)

    def forward(self, x):
        x = self.token_emb(x)
        x = x.transpose(1, 2)

        feats = []

        for conv in self.convs:
            h = F.relu(conv(x))
            h = F.max_pool1d(h, kernel_size=h.size(2)).squeeze(2)
            feats.append(h)
        h = torch.cat(feats, dim=1)
        h = self.dropout(h)

        return h

In [13]:
class PreferenceModel(nn.Module):
    def __init__(self, vocab_size, num_numeric_features, text_emb_dim=256, text_hidden_dim=128, kernel_sizes=(3, 4, 5), mlp_hidden=256, num_classes=3, dropout=0.2):
        super().__init__()

        self.text_encoder = TextCNNEncoder(vocab_size=vocab_size, emb_dim=text_emb_dim, hidden_dim=text_hidden_dim, kernel_sizes=kernel_sizes, dropout=dropout)
        text_dim = self.text_encoder.out_dim
        final_dim = text_dim + text_dim + text_dim + text_dim + text_dim  + num_numeric_features

        self.mlp = nn.Sequential(
            nn.Linear(final_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden // 2, num_classes)
        )

    def forward(self, prompt_ids, resp_a_ids, resp_b_ids, numeric_features):
        h_promt = self.text_encoder(prompt_ids)
        h_a = self.text_encoder(resp_a_ids)
        h_b = self.text_encoder(resp_b_ids)


        z = torch.cat([h_promt, h_a, h_b, torch.abs(h_a - h_b), h_a * h_b, numeric_features], dim=1)

        logits = self.mlp(z)
        return logits


In [14]:
def evaluate(model, loader, criterion, device):
    model.eval()
    losses = []
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            prompt_ids = batch["prompt_ids"].to(device)
            a_ids = batch["a_ids"].to(device)
            b_ids = batch["b_ids"].to(device)
            numeric_features = batch["numeric_features"].to(device)
            labels = batch["label"].to(device)

            logits = model(
                prompt_ids=prompt_ids,
                resp_a_ids=a_ids,
                resp_b_ids=b_ids,
                numeric_features=numeric_features
            )

            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return sum(losses) / len(losses), correct / total

In [19]:
EPOCHS = 20
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = PreferenceModel(len(vocab), num_numeric_features=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=EPOCHS)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    mean_loss = []

    for batch in train_loader:
        prompts_ids = batch["prompt_ids"].to(device)
        resp_a_ids = batch["a_ids"].to(device)
        resp_b_ids = batch["b_ids"].to(device)
        numeric_features = batch["numeric_features"].to(device)

        labels = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(prompts_ids, resp_a_ids, resp_b_ids, numeric_features)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        mean_loss.append(loss.item())

    scheduler.step()

    train_loss = sum(mean_loss) / len(mean_loss)
    
    print(f"EPOCH {epoch + 1}/{EPOCHS}, loss={train_loss:.4f}")

RuntimeError: MPS backend out of memory (MPS allocated: 4.02 GiB, other allocations: 16.07 GiB, max allowed: 20.13 GiB). Tried to allocate 293.30 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).